# US Stock Selection Algorithm

This notebook implements an algorithm to select potential US stocks to invest in from the S&P 500 index.

## Strategy
The algorithm screens stocks based on the following criteria:
1. **Trend**: The stock price is above its 200-day Simple Moving Average (SMA), indicating a long-term uptrend.
2. **Momentum**: A "Golden Cross" condition where the 50-day SMA is above the 200-day SMA.
3. **Valuation/Overbought**: The Relative Strength Index (RSI) is below 70, suggesting the stock is not currently overbought.


In [ ]:
# Install necessary libraries
!pip install yfinance pandas matplotlib requests beautifulsoup4 lxml

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import requests
import io

## 1. Get List of S&P 500 Tickers
We scrape the list of S&P 500 companies from Wikipedia.

In [ ]:
def get_sp500_tickers():
    """Scrapes S&P 500 tickers from Wikipedia."""
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        
        # Use pandas to read tables from the HTML content
        tables = pd.read_html(io.StringIO(response.text))
        
        # The first table is usually the S&P 500 components
        df = tables[0]
        if 'Symbol' in df.columns:
            tickers = df['Symbol'].tolist()
            # Replace dots with hyphens for yfinance (e.g. BRK.B -> BRK-B)
            tickers = [ticker.replace('.', '-') for ticker in tickers]
            return tickers
        else:
            print("Could not find the expected table structure.")
            return []
            
    except Exception as e:
        print(f"Error scraping tickers: {e}")
        return []

all_tickers = get_sp500_tickers()
print(f"Found {len(all_tickers)} tickers.")
print(f"Sample: {all_tickers[:10]}")

## 2. Define Analysis Functions
We define functions to calculate RSI and fetch stock data with moving averages.

In [ ]:
def calculate_rsi(data, window=14):
    """Calculates the Relative Strength Index (RSI)."""
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

def analyze_stock(ticker):
    """Fetches data and calculates indicators for a single stock."""
    try:
        # Download data for the last 2 years
        df = yf.download(ticker, period="2y", progress=False)
        
        if df.empty:
            return None
        
        # Handle MultiIndex columns if present (yfinance update)
        if isinstance(df.columns, pd.MultiIndex):
             try:
                 close = df['Close'][ticker]
             except KeyError:
                 close = df['Close'] # Fallback
        else:
            close = df['Close']
            
        # Ensure close is a Series
        if isinstance(close, pd.DataFrame):
             close = close.iloc[:, 0]

        # Calculate Moving Averages
        sma_50 = close.rolling(window=50).mean()
        sma_200 = close.rolling(window=200).mean()
        
        # Calculate RSI
        rsi = calculate_rsi(close)
        
        # Create a new DataFrame with the calculated values
        result_df = pd.DataFrame(index=df.index)
        result_df['Close'] = close
        result_df['SMA_50'] = sma_50
        result_df['SMA_200'] = sma_200
        result_df['RSI'] = rsi
        
        return result_df
    except Exception as e:
        print(f"Error analyzing {ticker}: {e}")
        return None

## 3. Run Analysis
We iterate through the tickers and apply our filtering criteria. For demonstration, we'll limit to the first 50 tickers to save time. Remove the limit to run on all stocks.

In [ ]:
selected_stocks = []

# LIMITING TO FIRST 50 FOR DEMO SPEED
# Set tickers_to_scan = all_tickers to scan everything
tickers_to_scan = all_tickers[:50] 

print(f"Scanning {len(tickers_to_scan)} tickers...")

for ticker in tickers_to_scan:
    # print(f"Processing {ticker}...", end=" ")
    df = analyze_stock(ticker)
    
    if df is not None and len(df) > 200:
        last_row = df.iloc[-1]
        
        # Strategy Logic
        if (last_row['Close'] > last_row['SMA_200'] and 
            last_row['RSI'] < 70 and 
            last_row['SMA_50'] > last_row['SMA_200']):
            
            selected_stocks.append({
                'Ticker': ticker,
                'Close': last_row['Close'],
                'SMA_50': last_row['SMA_50'],
                'SMA_200': last_row['SMA_200'],
                'RSI': last_row['RSI']
            })

print(f"\nSelected {len(selected_stocks)} stocks.")

## 4. Display Results
Show the selected stocks sorted by RSI (lower RSI might indicate better entry point within the uptrend).

In [ ]:
results_df = pd.DataFrame(selected_stocks)

if not results_df.empty:
    display_df = results_df.sort_values(by='RSI').round(2)
    print(display_df.to_string(index=False))
else:
    print("No stocks matched the criteria.")

## 5. Visual Inspection
Plot the chart for the top candidate.

In [ ]:
if not results_df.empty:
    top_pick = results_df.sort_values(by='RSI').iloc[0]['Ticker']
    print(f"\nGenerating chart for top pick: {top_pick}")
    
    df = analyze_stock(top_pick)
    
    plt.figure(figsize=(12, 8))
    plt.plot(df.index, df['Close'], label='Close Price', alpha=0.5)
    plt.plot(df.index, df['SMA_50'], label='SMA 50', color='orange')
    plt.plot(df.index, df['SMA_200'], label='SMA 200', color='red')
    plt.title(f'{top_pick} Price Analysis')
    plt.xlabel('Date')
    plt.ylabel('Price')
    plt.legend()
    plt.grid(True)
    plt.show()